In [1]:
## 1. ASK
## How many days last season had optimal conditions for snowmaking?
## Estimate the electricity budget required to run all snowmaking equipment for every hour with optimal conditions.
## Graph all days of the season using a heatmap.
## Color all days with a gradient according to the conditions for snowmaking. Use facets to separate each month.
## Graph all snow that fell naturally last season using a line.

## 2a. PREPARE / RESEARCH
## The winter season is November 1, 2024 - March 31, 2025
## Assuming trails open at 9AM and close at 4:30PM
## Snowmaking crews work for 16.5 hours
## To make snow, temperature must be 27 degrees Farenheit and below
## Humidity must be low between 30% and 60%
## Snowshoe mountain is equipped with 75 DemacLenko Titan 2.0 fixed position tower fan guns.
## 90 low-energy "stick" guns which use 10% of the energy.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import sys
print(sys.version)
print('pandas version', pd.__version__)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
pandas version 2.3.3


In [4]:
## This dataset provided by the NCEI or National Centers for Environmental Information has a record of daily snowfall totals.
## As provided, this table has a title. I'm going to include a parameter header=1 to begin the index on the second row.

In [5]:
sn = pd.read_csv("C:/Users/Spencer/code/github/resort-snowmaking-energy-budget/data/raw/data(9).csv", header=1)

In [6]:
sn

,Date,TAVG (Degrees Fahrenheit),TMAX (Degrees Fahrenheit),TMIN (Degrees Fahrenheit),PRCP (Inches),SNOW (Inches),SNWD (Inches)
0,1975-02-01,NaN,40.0,31.0,0.46,0.0,0.0
1,1975-02-02,NaN,34.0,27.0,0.40,4.0,4.0
2,1975-02-03,NaN,36.0,19.0,0.00,0.0,NaN
3,1975-02-04,NaN,30.0,14.0,0.00,0.0,NaN
4,1975-02-05,NaN,45.0,17.0,1.18,15.0,17.0
...,...,...,...,...,...,...,...
18571,2025-12-06,NaN,27.0,18.0,0.01,0.5,3.0
18572,2025-12-07,NaN,30.0,17.0,0.00,0.0,NaN
18573,2025-12-08,NaN,35.0,19.0,0.01,0.2,3.0
18574,2025-12-09,NaN,26.0,14.0,0.30,2.0,5.0


In [7]:
sn.columns

Index(['Date', 'TAVG (Degrees Fahrenheit)', 'TMAX (Degrees Fahrenheit)',
       'TMIN (Degrees Fahrenheit)', 'PRCP (Inches)', 'SNOW (Inches)',
       'SNWD (Inches)'],
      dtype='object')

In [8]:
sn[['Date', 'SNOW (Inches)']]

,Date,SNOW (Inches)
0,1975-02-01,0.0
1,1975-02-02,4.0
2,1975-02-03,0.0
3,1975-02-04,0.0
4,1975-02-05,15.0
...,...,...
18571,2025-12-06,0.5
18572,2025-12-07,0.0
18573,2025-12-08,0.2
18574,2025-12-09,2.0


In [9]:
sn_ss = sn[['Date', 'SNOW (Inches)']]

In [10]:
## This dataset is 18,576 rows long
## I only need the '24-'25 winter season as a subset
## Start of season is November 11
## End of season is March 30

In [11]:
sn_ss.query("Date >= '2024-11-01' and Date <= '2025-04-30'")

,Date,SNOW (Inches)
18171,2024-11-01,0.0
18172,2024-11-02,0.0
18173,2024-11-03,0.0
18174,2024-11-04,0.0
18175,2024-11-05,0.0
...,...,...
18347,2025-04-26,0.0
18348,2025-04-27,0.0
18349,2025-04-28,0.0
18350,2025-04-29,0.0


In [12]:
sn_ss = sn_ss.query("Date >= '2024-11-01' and Date <= '2025-04-30'")

In [13]:
## 17 days are missing daily snowfall

In [14]:
sn_ss.isna().sum()

Date              0
SNOW (Inches)    17
dtype: int64

In [15]:
sn_ss.query('`SNOW (Inches)`.isna()')

,Date,SNOW (Inches)
18209,2024-12-09,NaN
18210,2024-12-10,NaN
18211,2024-12-11,NaN
18213,2024-12-13,NaN
18215,2024-12-15,NaN
18217,2024-12-17,NaN
18218,2024-12-18,NaN
18219,2024-12-19,NaN
18225,2024-12-25,NaN
18226,2024-12-26,NaN


In [16]:
sn_ss = sn_ss.fillna(0)

In [17]:
sn_ss.isna().sum()

Date             0
SNOW (Inches)    0
dtype: int64

In [18]:
## There was no snow on Christmas!

In [19]:
sn_ss.loc[sn_ss['Date'] == '2024-12-25']

,Date,SNOW (Inches)
18225,2024-12-25,0.0


In [20]:
## 128.4 inches of natural snow during the '24-'25 winter season.

In [21]:
sn_ss['SNOW (Inches)'].sum()

np.float64(128.4)

In [22]:
## This is a dataset from WeatherUnderground, they keep historical weather data from the timeframe we need.
## The reason I chose this dataset is the humidity column.
## The data is also very precise, with observations being roughly an hour apart.

In [23]:
el = pd.read_csv("C:/Users/Spencer/code/github/resort-snowmaking-energy-budget/data/raw/Elkins-Randolph County Regional Airport Station - Sheet1.csv")

In [24]:
el.shape

(5230, 11)

In [25]:
el.head()

,Date,Time,Temperature,Dew Point,Humidity,Wind,Wind Speed,Wind Gust,Pressure,Precip.,Condition
0,11-01-2024,11:51 PM,68 °F,51 °F,55 %,S,7 mph,0 mph,27.98 in,0.0 in,Cloudy
1,11-01-2024,12:51 AM,67 °F,52 °F,59 %,S,9 mph,0 mph,27.98 in,0.0 in,Cloudy
2,11-01-2024,1:51 AM,67 °F,53 °F,61 %,S,6 mph,0 mph,27.97 in,0.0 in,Cloudy
3,11-01-2024,2:51 AM,66 °F,56 °F,70 %,W,7 mph,0 mph,27.98 in,0.0 in,Cloudy
4,11-01-2024,3:51 AM,65 °F,57 °F,75 %,WSW,6 mph,0 mph,27.99 in,0.0 in,Light Rain


In [26]:
el.dtypes

Date           object
Time           object
Temperature    object
Dew Point      object
Humidity       object
Wind           object
Wind Speed     object
Wind Gust      object
Pressure       object
Precip.        object
Condition      object
dtype: object

In [27]:
el.columns

Index(['Date', 'Time', 'Temperature', 'Dew Point', 'Humidity', 'Wind',
       'Wind Speed', 'Wind Gust', 'Pressure', 'Precip.', 'Condition'],
      dtype='object')

In [28]:
## Copy and paste the results of el.columns into double brackets to take a subset of the data
## Rob Mulla suggests commenting the extraneous columns, if you need them later

In [29]:
el[['Date', 'Time', 'Temperature',
    #'Dew Point',
    'Humidity',
    #'Wind', 'Wind Speed', 'Wind Gust', 'Pressure', 'Precip.', 'Condition'
   ]]

,Date,Time,Temperature,Humidity
0,11-01-2024,11:51 PM,68 °F,55 %
1,11-01-2024,12:51 AM,67 °F,59 %
2,11-01-2024,1:51 AM,67 °F,61 %
3,11-01-2024,2:51 AM,66 °F,70 %
4,11-01-2024,3:51 AM,65 °F,75 %
...,...,...,...,...
5225,03-31-2025,8:51 PM,49 °F,97 %
5226,03-31-2025,9:01 PM,49 °F,97 %
5227,03-31-2025,9:30 PM,47 °F,93 %
5228,03-31-2025,9:51 PM,46 °F,93 %


In [30]:
el['Datetime_Col'] = el['Date'] + ' ' + el['Time']

In [31]:
#el['Datetime_Col'] = pd.to_datetime(el['Datetime_Col'])

In [36]:
el['Datetime_Col'] = pd.to_datetime(el['Datetime_Col'], format='%m-%d-%Y %I:%M %p')

In [37]:
el['hours_since_last'] = el.groupby(el['Datetime_Col'].dt.date)['Datetime_Col'].diff().fillna(pd.Timedelta(seconds=0))

In [38]:
el['hours_since_last'] = el['hours_since_last'].dt.total_seconds() / 3600

In [39]:
el['Week_Number'] = el['Datetime_Col'].dt.isocalendar().week

In [40]:
el['day_of_week_num'] = el['Datetime_Col'].dt.weekday

In [41]:
el.to_csv("C:/Users/Spencer/code/github/resort-snowmaking-energy-budget/data/processed/Elkins_Subset_2026_02_17_v00.csv")

In [42]:
el

,Date,Time,Temperature,Dew Point,Humidity,Wind,Wind Speed,Wind Gust,Pressure,Precip.,Condition,Datetime_Col,hours_since_last,Week_Number,day_of_week_num
0,11-01-2024,11:51 PM,68 °F,51 °F,55 %,S,7 mph,0 mph,27.98 in,0.0 in,Cloudy,2024-11-01 23:51:00,0.000000,44,4
1,11-01-2024,12:51 AM,67 °F,52 °F,59 %,S,9 mph,0 mph,27.98 in,0.0 in,Cloudy,2024-11-01 00:51:00,-23.000000,44,4
2,11-01-2024,1:51 AM,67 °F,53 °F,61 %,S,6 mph,0 mph,27.97 in,0.0 in,Cloudy,2024-11-01 01:51:00,1.000000,44,4
3,11-01-2024,2:51 AM,66 °F,56 °F,70 %,W,7 mph,0 mph,27.98 in,0.0 in,Cloudy,2024-11-01 02:51:00,1.000000,44,4
4,11-01-2024,3:51 AM,65 °F,57 °F,75 %,WSW,6 mph,0 mph,27.99 in,0.0 in,Light Rain,2024-11-01 03:51:00,1.000000,44,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5225,03-31-2025,8:51 PM,49 °F,48 °F,97 %,NW,13 mph,22 mph,27.71 in,0.0 in,Light Rain,2025-03-31 20:51:00,0.083333,14,0
5226,03-31-2025,9:01 PM,49 °F,48 °F,97 %,NW,13 mph,0 mph,27.71 in,0.0 in,Light Rain,2025-03-31 21:01:00,0.166667,14,0
5227,03-31-2025,9:30 PM,47 °F,45 °F,93 %,NW,10 mph,23 mph,27.73 in,0.0 in,Light Rain,2025-03-31 21:30:00,0.483333,14,0
5228,03-31-2025,9:51 PM,46 °F,44 °F,93 %,WNW,8 mph,23 mph,27.75 in,0.0 in,Light Rain,2025-03-31 21:51:00,0.350000,14,0


In [43]:
#el = el[['Date', 'Time', 'Temperature', 'Humidity']]

In [44]:
el = el[['Temperature', 'Humidity', 'Datetime_Col', 'hours_since_last', 'Week_Number', 'day_of_week_num']]

In [45]:
## Remember, the conditions for making snow are a temperature of 27 degrees farenheit and below Humidity must be between 30 and 60 percent
## This is every observation of the season where snowmaking was possible

In [46]:
el.query("Temperature <= '27' and Humidity >= '30' and Humidity <= '60'")

,Temperature,Humidity,Datetime_Col,hours_since_last,Week_Number,day_of_week_num
1114,24 °F,57 %,2024-11-30 17:51:00,1.000000,48,5
1115,24 °F,55 %,2024-11-30 18:51:00,1.000000,48,5
1169,22 °F,58 %,2024-12-01 21:51:00,1.000000,48,6
1337,25 °F,55 %,2024-12-07 09:51:00,1.000000,49,5
1543,26 °F,58 %,2024-12-12 06:51:00,0.666667,50,3
...,...,...,...,...,...,...
4420,22 °F,53 %,2025-03-02 09:51:00,1.000000,9,6
4421,25 °F,46 %,2025-03-02 10:51:00,1.000000,9,6
4430,25 °F,51 %,2025-03-02 19:51:00,1.000000,9,6
4431,24 °F,55 %,2025-03-02 20:51:00,1.000000,9,6


In [47]:
## Save this query as a subset of the original dataset without modifying the original

In [48]:
el_ss = el.query("Temperature <= '27' and Humidity >= '30' and Humidity <= '60'")

In [49]:
el_ss.dtypes

Temperature                 object
Humidity                    object
Datetime_Col        datetime64[ns]
hours_since_last           float64
Week_Number                 UInt32
day_of_week_num              int32
dtype: object

In [50]:
#el_ss['Datetime_Col'] = el_ss['Date'] + ' ' + el_ss['Time']

In [51]:
el_ss

,Temperature,Humidity,Datetime_Col,hours_since_last,Week_Number,day_of_week_num
1114,24 °F,57 %,2024-11-30 17:51:00,1.000000,48,5
1115,24 °F,55 %,2024-11-30 18:51:00,1.000000,48,5
1169,22 °F,58 %,2024-12-01 21:51:00,1.000000,48,6
1337,25 °F,55 %,2024-12-07 09:51:00,1.000000,49,5
1543,26 °F,58 %,2024-12-12 06:51:00,0.666667,50,3
...,...,...,...,...,...,...
4420,22 °F,53 %,2025-03-02 09:51:00,1.000000,9,6
4421,25 °F,46 %,2025-03-02 10:51:00,1.000000,9,6
4430,25 °F,51 %,2025-03-02 19:51:00,1.000000,9,6
4431,24 °F,55 %,2025-03-02 20:51:00,1.000000,9,6


In [52]:
#el_ss['Datetime_Col'] = pd.to_datetime(el_ss['Datetime_Col'])

In [53]:
el_ss.dtypes

Temperature                 object
Humidity                    object
Datetime_Col        datetime64[ns]
hours_since_last           float64
Week_Number                 UInt32
day_of_week_num              int32
dtype: object

In [54]:
#el_ss.sort_values('Datetime_Col')

In [55]:

el_ss['hours_since_last'] = el_ss.groupby(el_ss['Datetime_Col'].dt.date)['Datetime_Col'].diff().fillna(pd.Timedelta(seconds=0))

C:\Users\Spencer\AppData\Local\Temp\ipykernel_15284\477777036.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  el_ss['hours_since_last'] = el_ss.groupby(el_ss['Datetime_Col'].dt.date)['Datetime_Col'].diff().fillna(pd.Timedelta(seconds=0))


In [56]:
#el_ss['hours_since_last'] = el_ss['hours_since_last'].dt.total_seconds() / 3600

In [57]:
el_ss

,Temperature,Humidity,Datetime_Col,hours_since_last,Week_Number,day_of_week_num
1114,24 °F,57 %,2024-11-30 17:51:00,0 days 00:00:00,48,5
1115,24 °F,55 %,2024-11-30 18:51:00,0 days 01:00:00,48,5
1169,22 °F,58 %,2024-12-01 21:51:00,0 days 00:00:00,48,6
1337,25 °F,55 %,2024-12-07 09:51:00,0 days 00:00:00,49,5
1543,26 °F,58 %,2024-12-12 06:51:00,0 days 00:00:00,50,3
...,...,...,...,...,...,...
4420,22 °F,53 %,2025-03-02 09:51:00,0 days 01:00:00,9,6
4421,25 °F,46 %,2025-03-02 10:51:00,0 days 01:00:00,9,6
4430,25 °F,51 %,2025-03-02 19:51:00,0 days 09:00:00,9,6
4431,24 °F,55 %,2025-03-02 20:51:00,0 days 01:00:00,9,6


In [58]:
el_ss['hours_since_last'].sum()

Timedelta('3 days 02:00:00')

In [59]:
## There were 74 hours in the season where conditions were optimal for snowmaking
## I'm saving this value as a variable for the electricity budget estimate

In [60]:
op_hr = el_ss['hours_since_last'].sum()

In [61]:
total_months = el_ss['Datetime_Col'].dt.month.nunique()

In [62]:
total_months

5

In [63]:
#el_ss['Week_Number'] = el_ss['Datetime_Col'].dt.isocalendar().week

In [64]:
#el_ss['day_of_week_name'] = el_ss['Datetime_Col'].dt.day_name()

In [65]:
el_ss

,Temperature,Humidity,Datetime_Col,hours_since_last,Week_Number,day_of_week_num
1114,24 °F,57 %,2024-11-30 17:51:00,0 days 00:00:00,48,5
1115,24 °F,55 %,2024-11-30 18:51:00,0 days 01:00:00,48,5
1169,22 °F,58 %,2024-12-01 21:51:00,0 days 00:00:00,48,6
1337,25 °F,55 %,2024-12-07 09:51:00,0 days 00:00:00,49,5
1543,26 °F,58 %,2024-12-12 06:51:00,0 days 00:00:00,50,3
...,...,...,...,...,...,...
4420,22 °F,53 %,2025-03-02 09:51:00,0 days 01:00:00,9,6
4421,25 °F,46 %,2025-03-02 10:51:00,0 days 01:00:00,9,6
4430,25 °F,51 %,2025-03-02 19:51:00,0 days 09:00:00,9,6
4431,24 °F,55 %,2025-03-02 20:51:00,0 days 01:00:00,9,6


In [66]:
#el_ss.to_csv("C:/Users/Spencer Woo/Desktop/Elkins_Subset_2026_02_17_v00.csv")

In [67]:
## 2b. PREPARE / RESEARCH
## DemacLenko Titan 2.0 power consumption: 14-24 kW
## SnowLogic DV7
## In all, Snowshoe boasts 15 compressors
## Ingersroll Rand power consumption: 895 kW
## 10 pumps
## Average power consumption for a turbine pump: 930 kW
## Mon Power Commercial Rates: General service (Schedule B) includes an energy charge of $0.12808 per kWh for the first 400 kWh
## and $0.11403 for additional usage

In [68]:
#def sum_kw():
#    total_sum = 0
#    data = [
#        [24, 75],
#        [24 * .10, 90],
#        [930, 10],
#        [895, 15]
#    ]
#    for row in data:
#        product = row[0] * row[1]
#        total_sum += product
#    return total_sum

#def convert_kwh():
#    return sum_kw() * op_hr

#def estimate_budget(ra = 0.12808, kwha = 400, rb = 0.11403):
#    return f"${kwha * ra + (convert_kwh() - kwha) * rb:,.2f}"

#estimate_budget()

In [69]:
#data = {
#    'Tool': ['DemacLenko Titan 2.0 Fan Gun', 'SnowLogic DV7 Stick Gun', 'Turbine Pump', 'Ingersoll Rand Compressor'],
#    'Kilowatts': [24, 24 * .10, 930, 895],
#    'Quantity': [75, 90, 10, 15]
#}

In [70]:
#ni = pd.DataFrame(data)

In [71]:
#ni

In [72]:
#ni['Product'] = ni['Kilowatts'] * ni['Quantity']

In [73]:
#ni

In [74]:
#ni['Product'].sum()

In [75]:
#sum_kw = ni['Product'].sum()

In [76]:
#def convert_kwh():
#    return sum_kw * op_hr

In [77]:
#convert_kwh()

In [78]:
#def estimate_budget(ra = 0.12808, kwha = 400, rb = 0.11403):
#    return f"${kwha * ra + (convert_kwh() - kwha) * rb:,.2f}"

In [79]:
#estimate_budget()

In [80]:
## The hours_since_last column is not counting days with only one observation
## January 23, 25, February 14, December 1, 7

In [84]:
#el_ss.groupby('Date').size()

In [85]:
el_ss.groupby('Datetime_Col').size()

Datetime_Col
2024-11-30 17:51:00    1
2024-11-30 18:51:00    1
2024-12-01 21:51:00    1
2024-12-07 09:51:00    1
2024-12-12 06:51:00    1
                      ..
2025-12-18 17:51:00    1
2025-12-18 18:51:00    1
2025-12-18 19:51:00    1
2025-12-18 20:51:00    1
2025-12-18 21:51:00    1
Length: 61, dtype: int64

In [86]:
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columnns', None)
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')

In [87]:
inventory = np.array([['Tool', 'Kilowatts', 'Quantity'],
                 ['DemacLenko Titan 2.0 Fan Gun', 24, 75],
                 ['SnowLogic DV7 Stick Gun', 24 * .10, 90],
                  ['Turbine Pump', 930, 10],
                  ['Ingersoll Rand Compressor', 895, 15]])


In [88]:
print(inventory.shape)

(5, 3)


In [89]:
print(inventory[0:])

[['Tool' 'Kilowatts' 'Quantity']
 ['DemacLenko Titan 2.0 Fan Gun' '24' '75']
 ['SnowLogic DV7 Stick Gun' '2.4000000000000004' '90']
 ['Turbine Pump' '930' '10']
 ['Ingersoll Rand Compressor' '895' '15']]


In [90]:
print(inventory[1:,1])

['24' '2.4000000000000004' '930' '895']


In [91]:
print(inventory[1:,2])

['75' '90' '10' '15']


In [92]:
x = inventory[1:,1].astype(float)
y = inventory[1:,2].astype(float)

In [93]:
np.multiply(x, y)

array([ 1800.,   216.,  9300., 13425.])

In [94]:
np.sum(np.multiply(x,y))

np.float64(24741.0)

In [95]:
sum_kw = np.sum(np.multiply(x,y))

In [96]:
#def convert_kwh():
 #   return sum_kw * op_hr

#def estimate_budget(charge = 200, block = 400, first = 0.12808, over = 0.11403):
#    return f"${charge + block * first + (convert_kwh() - block) * over:,.2f}"

In [97]:
#estimate_budget(200, 500, 18.20, 16.48)

In [98]:
#sum_kw

In [99]:
#def demand_charge(block = 500, first = 18.20, over = 16.48):
#    return block * first + (sum_kw - block) * over
#def energy_charge(flat = 0.05146):
#    return convert_kwh() * flat
#def estimate_budget(charge = 200):
#    return f"${charge + demand_charge() + energy_charge():,.2f}"

In [100]:
#demand_charge()

In [101]:
#energy_charge()

In [102]:
#estimate_budget()

In [103]:
def customer_charge(charge = 10):
    return charge * total_months
def convert_kwh():
    return sum_kw * op_hr
def energy_charge(block = 400, first = 0.12808, over = 0.11403):
    return block * first + (convert_kwh() - block) * over
def estimate_budget():
    return f"${customer_charge() + energy_charge():,.2f}"

In [104]:
estimate_budget()

TypeError: unsupported operand type(s) for -: 'Timedelta' and 'int'